# LZW 与 LZSS 编码演示

这个 notebook 直接复用 `src/encoder.py` 中的实现，展示两种算法对文本和图像字节流的编码、解码与压缩效果。

说明：这里的压缩率是基于结构化输出的近似估算，不是最终二进制封包格式。

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.encoder import LZWEncoder, LZSSEncoder, LiteralToken, MatchToken

lzw = LZWEncoder()
lzss = LZSSEncoder()

In [ ]:
def lzw_bits(codes: list[int]) -> int:
    if not codes:
        return 0
    width = max(9, math.ceil(math.log2(max(codes) + 1)))
    return len(codes) * width


def lzss_bits(tokens: list[LiteralToken | MatchToken], encoder: LZSSEncoder) -> int:
    offset_bits = max(1, math.ceil(math.log2(encoder.window_size + 1)))
    length_bits = max(1, math.ceil(math.log2(encoder.lookahead_size + 1)))
    total = 0
    for token in tokens:
        if isinstance(token, LiteralToken):
            total += 1 + 8
        else:
            total += 1 + offset_bits + length_bits
    return total


def summarize_case(name: str, data: bytes) -> dict[str, float | int | str]:
    lzw_codes = lzw.encode(data)
    lzss_tokens = lzss.encode(data)

    assert lzw.decode(lzw_codes) == data
    assert lzss.decode(lzss_tokens) == data

    original_bits = len(data) * 8
    lzw_total = lzw_bits(lzw_codes)
    lzss_total = lzss_bits(lzss_tokens, lzss)

    def ratio(bits: int) -> float:
        return bits / original_bits if original_bits else 0.0

    return {
        'case': name,
        'input_bytes': len(data),
        'lzw_units': len(lzw_codes),
        'lzss_units': len(lzss_tokens),
        'lzw_ratio': ratio(lzw_total),
        'lzss_ratio': ratio(lzss_total),
    }


def summary_frame(rows: list[dict[str, float | int | str]]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    return frame.assign(
        lzw_ratio=frame['lzw_ratio'].map(lambda value: round(float(value), 3)),
        lzss_ratio=frame['lzss_ratio'].map(lambda value: round(float(value), 3)),
    )


def preview_lzss(tokens: list[LiteralToken | MatchToken], limit: int = 16) -> pd.DataFrame:
    rows = []
    for index, token in enumerate(tokens[:limit]):
        if isinstance(token, LiteralToken):
            char = chr(token.value) if 32 <= token.value <= 126 else '.'
            rows.append({
                'index': index,
                'kind': 'literal',
                'value': token.value,
                'char': char,
                'offset': None,
                'length': None,
            })
        else:
            rows.append({
                'index': index,
                'kind': 'match',
                'value': None,
                'char': None,
                'offset': token.offset,
                'length': token.length,
            })
    return pd.DataFrame(rows)


def show_grayscale(image: np.ndarray, pixel_size: int = 12) -> None:
    rows = []
    for row in image:
        cells = []
        for value in row:
            shade = int(value)
            cells.append(
                f"<td style='width:{pixel_size}px;height:{pixel_size}px;padding:0;background:rgb({shade},{shade},{shade})'></td>"
            )
        rows.append('<tr>' + ''.join(cells) + '</tr>')
    display(HTML(
        "<table style='border-collapse:collapse;border:1px solid #999'>" + ''.join(rows) + "</table>"
    ))

## 文本示例

LZW 对重复短语通常比较敏感；LZSS 则倾向把重复片段表示成回溯引用。

In [ ]:
text = (
    "TOBEORNOTTOBEORTOBEORNOT\n"
    "LZW and LZSS both exploit repeated patterns. \
Repeated patterns make compression easier.\n"
) * 4
text_bytes = text.encode('utf-8')

lzw_codes = lzw.encode(text_bytes)
lzss_tokens = lzss.encode(text_bytes)

print('Original text preview:')
print(text[:140])
print()
print('First 24 LZW codes:')
print(lzw_codes[:24])
print()
display(preview_lzss(lzss_tokens))
display(summary_frame([summarize_case('text', text_bytes)]))

## 图像示例

下面构造两个 32x32 的灰度图像：

- `structured_image`：规则条纹和块状重复，适合压缩
- `noisy_image`：随机噪声，通常不容易压缩

In [ ]:
structured_image = np.zeros((32, 32), dtype=np.uint8)
structured_image[:, ::4] = 255
structured_image[8:24, 8:24] = 180
structured_image[12:20, 12:20] = 40

rng = np.random.default_rng(7)
noisy_image = rng.integers(0, 256, size=(32, 32), dtype=np.uint8)

print('Structured image')
show_grayscale(structured_image)
print('Noisy image')
show_grayscale(noisy_image)

In [ ]:
structured_bytes = structured_image.tobytes()
noisy_bytes = noisy_image.tobytes()

rows = [
    summarize_case('structured_image', structured_bytes),
    summarize_case('noisy_image', noisy_bytes),
]
display(summary_frame(rows))

## 观察

- 对重复明显的文本，LZW 和 LZSS 都能减少表示长度。
- 对规则图像，压缩效果通常比随机噪声更好。
- 对噪声图像，结构化压缩往往接近无效，甚至可能变差。
- 如果要做更真实的压缩率比较，下一步应把结构化结果打包成真正的二进制位流。